In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [57]:
data = pd.read_csv("d:\\UAH GO CHARGERS\\Spring 26\\MSC450\\Module 6\\election2000-1.csv")
data.head()

,Bush,UnE,Pop,Male,Male18,Pop65,NonMet,Pov,NuHouse,Inc50,Inc75,Inc100
0,56.47,4.6,4447100,48.27,47.26,13.0,32.68,16.1,1472906,14.3,4.3,2.0
1,58.62,6.6,626932,51.69,51.83,5.7,58.48,9.4,85359,36.3,16.8,6.9
2,51.02,3.9,5130632,49.92,49.42,13.0,14.03,13.9,1571330,11.0,4.2,1.8
3,51.31,4.4,2673400,48.80,47.94,14.0,53.66,15.8,836388,10.0,2.9,1.6
4,41.65,4.9,33871648,49.82,49.26,10.6,3.31,14.2,9709296,23.1,10.0,4.4


In [58]:
#Model 1

In [59]:
import statsmodels.api as sm
ols = sm.OLS(data.iloc[:,0], sm.add_constant(data.iloc[:,1:]))
lm1 = ols.fit()
print(lm1.summary())

                            OLS Regression Results                            
Dep. Variable:                   Bush   R-squared:                       0.815
Model:                            OLS   Adj. R-squared:                  0.755
Method:                 Least Squares   F-statistic:                     13.61
Date:                Thu, 26 Feb 2026   Prob (F-statistic):           1.89e-09
Time:                        12:07:58   Log-Likelihood:                -133.98
No. Observations:                  46   AIC:                             292.0
Df Residuals:                      34   BIC:                             313.9
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       -652.3175    191.613     -3.404      0.0

In [60]:
#Model 2

In [61]:
import itertools
def fit_lm(feature_set):
    ols = sm.OLS(data.Bush, sm.add_constant(data[list(feature_set)]))
    lm = ols.fit()
    RSS = ((lm.predict(sm.add_constant(data[list(feature_set)])) - data.Bush) ** 2).sum()
    rsquared_adj = lm.rsquared_adj
    AIC = lm.aic
    BIC = lm.bic
    return {"model":lm, "RSS":RSS,"Rsquare_adj": rsquared_adj,"AIC":AIC,"BIC":BIC}

def getbest(k):
    result = []
    for combo in itertools.combinations(data.columns[1:], k):
        result.append(fit_lm(combo))
    models = pd.DataFrame(result)
    best_model = models.loc[models["BIC"].argmin()]
    return best_model

In [62]:
import time
models_best = pd.DataFrame(columns=["model","RSS","Rsquared_adj","AIC","BIC"])
start = time.time()
for i in range(1,len(data.columns)):
    models_best.loc[i] = getbest(i)
end = time.time()
print(models_best)
print("Best subset selection takes: ", end-start, "seconds")
best_index2 = models_best["BIC"].idxmin()
lm2 = models_best.loc[best_index2,"model"]
print(lm2.summary())

                                                model          RSS  \
1   <statsmodels.regression.linear_model.Regressio...  2969.238063   
2   <statsmodels.regression.linear_model.Regressio...  2166.632919   
3   <statsmodels.regression.linear_model.Regressio...  1384.370173   
4   <statsmodels.regression.linear_model.Regressio...  1166.068477   
5   <statsmodels.regression.linear_model.Regressio...  1052.406013   
6   <statsmodels.regression.linear_model.Regressio...    986.58975   
7   <statsmodels.regression.linear_model.Regressio...   937.245641   
8   <statsmodels.regression.linear_model.Regressio...    922.92846   
9   <statsmodels.regression.linear_model.Regressio...   912.863033   
10  <statsmodels.regression.linear_model.Regressio...   912.467606   
11  <statsmodels.regression.linear_model.Regressio...   912.461597   

   Rsquared_adj         AIC         BIC  
1           NaN  326.243631  329.900914  
2           NaN  313.747602  319.233526  
3           NaN  295.142867  302.

In [63]:
#Model 3 Forward Selection

In [64]:
remain = list(data.columns[1:])
picked = list()
model_best1 = pd.DataFrame(columns=["model","RSS","Rsquared_adj","AIC","BIC"])
start = time.time()
for i in range(1,len(data.columns)):
    best_BIC = np.inf
    for combo in itertools.combinations(remain,1):
        BIC = fit_lm(list(combo) + picked)["BIC"]
        if BIC < best_BIC:
            best_BIC = BIC
            best_feature = combo[0]
            model_best1.loc[i] = fit_lm(list(combo) + picked)
    picked.append(best_feature)
    remain.remove(best_feature)

end = time.time()
best_index3 = model_best1["BIC"].idxmin()
lm3 = model_best1.loc[best_index3,"model"]
print(model_best1)
print("Foward selection take: ", end-start, "seconds")
print(lm3.summary())

                                                model          RSS  \
1   <statsmodels.regression.linear_model.Regressio...  2969.238063   
2   <statsmodels.regression.linear_model.Regressio...  2356.629542   
3   <statsmodels.regression.linear_model.Regressio...  1384.370173   
4   <statsmodels.regression.linear_model.Regressio...  1166.068477   
5   <statsmodels.regression.linear_model.Regressio...  1060.690673   
6   <statsmodels.regression.linear_model.Regressio...   986.589750   
7   <statsmodels.regression.linear_model.Regressio...   938.527163   
8   <statsmodels.regression.linear_model.Regressio...   928.362027   
9   <statsmodels.regression.linear_model.Regressio...   922.062471   
10  <statsmodels.regression.linear_model.Regressio...   912.651987   
11  <statsmodels.regression.linear_model.Regressio...   912.461597   

    Rsquared_adj         AIC         BIC  
1            NaN  326.243631  329.900914  
2            NaN  317.614276  323.100200  
3            NaN  295.142867  

In [65]:
#Model 4 Backward selection

In [66]:
pick = list(data.columns[1:])
model_best2 = pd.DataFrame(columns=["model","RSS","Rsquared_adj","AIC","BIC"])
start = time.time()
for i in range(1,len(data.columns)):
    best_BIC = np.inf
    for combo in itertools.combinations(pick,len(pick)-1):
        BIC = fit_lm(list(combo))["BIC"]
        if BIC < best_BIC:
            best_BIC = BIC
            best_feature = combo
            model_best2.loc[i] = fit_lm(list(combo))
    pick = best_feature
end = time.time()
best_index4 = model_best2["BIC"].idxmin()
lm4 = model_best2.loc[best_index4, "model"]
print(model_best2)
print("Backward selection takes: ", end-start, "seconds")
print(lm4.summary())

                                                model          RSS  \
1   <statsmodels.regression.linear_model.Regressio...   912.467606   
2   <statsmodels.regression.linear_model.Regressio...   912.863033   
3   <statsmodels.regression.linear_model.Regressio...   922.928460   
4   <statsmodels.regression.linear_model.Regressio...   937.245641   
5   <statsmodels.regression.linear_model.Regressio...   987.109192   
6   <statsmodels.regression.linear_model.Regressio...  1052.406013   
7   <statsmodels.regression.linear_model.Regressio...  1169.495053   
8   <statsmodels.regression.linear_model.Regressio...  1384.370173   
9   <statsmodels.regression.linear_model.Regressio...  2348.014847   
10  <statsmodels.regression.linear_model.Regressio...  3568.421052   
11  <statsmodels.regression.linear_model.Regressio...  4930.727765   

    Rsquared_adj         AIC         BIC  
1            NaN  289.967860  310.082915  
2            NaN  287.987790  306.274204  
3            NaN  286.492220  

In [67]:
#Model 5
#Apply forward selection then using K-fold corss validation to choose the best model

,Bush,UnE,Pop,Male,Male18,Pop65,NonMet,Pov,NuHouse,Inc50,Inc75,Inc100
0,62.25,3.0,1711263,49.29,48.61,13.6,47.42,9.7,599558,15.3,5.4,2.2
1,46.43,4.2,12281054,48.29,47.34,15.6,15.39,11.0,4256258,17.3,5.7,2.3
2,40.70,2.9,608827,49.01,48.27,12.7,67.33,9.4,190533,19.3,6.4,3.1
3,45.50,3.3,4919479,49.52,48.88,12.1,29.60,7.9,1876964,19.3,6.1,2.9
4,35.23,4.6,18976457,48.21,47.20,12.9,7.92,14.6,5862138,26.6,12.9,7.0


In [68]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

cv = KFold(n_splits=5, random_state=1, shuffle=True)
mse = []

for i in range(1,12):
    x = data[model_best1.loc[i,"model"].params.index[1:]]
    y = data.iloc[:,0]
    error = []

    for train_index, test_index in cv.split(data):
        x_train, x_test = x.iloc[train_index], x.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        ols = sm.OLS(y_train, sm.add_constant(x_train))
        lm = ols.fit()
        y_pred = lm.predict(sm.add_constant(x_test))
        error.append(mean_squared_error(y_pred,y_test))
    mse.append(np.mean(error))

model_best1["MSE"] = mse
best_index4 = model_best1["MSE"].idxmin()
lm5 = model_best1.loc[best_index4, "model"]

print(lm5.summary())

                            OLS Regression Results                            
Dep. Variable:                   Bush   R-squared:                       0.785
Model:                            OLS   Adj. R-squared:                  0.758
Method:                 Least Squares   F-statistic:                     29.19
Date:                Thu, 26 Feb 2026   Prob (F-statistic):           2.35e-12
Time:                        12:08:08   Log-Likelihood:                -137.45
No. Observations:                  46   AIC:                             286.9
Df Residuals:                      40   BIC:                             297.9
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       -775.2704    141.775     -5.468      0.0

In [69]:
test = pd.read_csv("d:\\UAH GO CHARGERS\\Spring 26\\MSC450\\Module 6\\election2000_test.csv")
test.head()

,Bush,UnE,Pop,Male,Male18,Pop65,NonMet,Pov,NuHouse,Inc50,Inc75,Inc100
0,62.25,3.0,1711263,49.29,48.61,13.6,47.42,9.7,599558,15.3,5.4,2.2
1,46.43,4.2,12281054,48.29,47.34,15.6,15.39,11.0,4256258,17.3,5.7,2.3
2,40.70,2.9,608827,49.01,48.27,12.7,67.33,9.4,190533,19.3,6.4,3.1
3,45.50,3.3,4919479,49.52,48.88,12.1,29.60,7.9,1876964,19.3,6.1,2.9
4,35.23,4.6,18976457,48.21,47.20,12.9,7.92,14.6,5862138,26.6,12.9,7.0


In [74]:
def predict_model(lm,test):
    predictors = lm.params.index[1:]
    
    x_test = test[predictors]
    x_test = sm.add_constant(x_test)
    
    y_test = test.iloc[:,0]
    
    y_pred = lm.predict(x_test)
    
    mse = mean_squared_error(y_test, y_pred)
    
    return mse
print("MSE for model 1: ",predict_model(lm1,test))
print("MSE for model 2: ",predict_model(lm2,test))
print("MSE for model 3: ",predict_model(lm3,test))
print("MSE for model 4: ",predict_model(lm4,test))
print("MSE for model 5: ",predict_model(lm5,test))

MSE for model 1:  83.04050107781043
MSE for model 2:  68.97138242923516
MSE for model 3:  76.37542126209755
MSE for model 4:  68.97138242923516
MSE for model 5:  76.37542126209755
